# M3 — Data quality and exploratory analysis

This notebook describes the pinned **development sample**, not an untouched model test set. It reads canonical records through the M1 adapters from immutable local archives. It does not download data, connect to PostgreSQL, fit preprocessing, or train models. Market values are editorial estimates, not transfer fees.

First run `python scripts/verify_ingestion.py` from the repository root to acquire the pinned inputs. Then restart the kernel and run all cells, or use `python scripts/execute_quality_notebook.py`. Reusable analysis lives in `pl_analytics.data.quality`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from pl_analytics.config import get_settings
from pl_analytics.data.quality import build_quality_report, load_quality_sample

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Run from the repository root or notebooks directory")
os.chdir(ROOT)
settings = get_settings()
manifest = Path(os.environ.get(
    "PL_ANALYTICS_QUALITY_MANIFEST", "data/manifests/m1_sources.json"
))
sample = load_quality_sample(manifest, settings.data_dir / "raw")
if sample.scope.competition_id not in settings.active_competitions:
    raise ValueError("Manifest competition is not enabled")
report = build_quality_report(sample)
output_dir = settings.artifact_dir / "m3"
output_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
display(report["scope"], report["counts"])
display(pd.DataFrame(report["provenance"])[["source", "sha256", "dataset_version", "retrieved_at"]])

## Coverage and missingness

These are **post-ingestion** records: rejected raw records and records outside the selected scope are not represented. A zero row count for appearances means a dataset was not loaded, not that players played zero minutes. Missing historical clubs are intentional; source-reported club context is not a verified historical assignment.

In [ ]:
missing = pd.concat(
    [pd.DataFrame(rows).assign(table=table) for table, rows in report["missingness"].items()],
    ignore_index=True,
)
display(missing[["table", "column", "rows", "missing", "missing_fraction"]])
display(report["duplicates"], report["leakage_and_coverage"])
coverage = pd.DataFrame({
    name: item["monthly_counts_utc"] for name, item in report["temporal"].items()
})
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, name, color in zip(axes, coverage.columns, ["#185A80", "#B15E24"], strict=True):
    ax.bar(coverage.index, coverage[name], color=color)
    ax.set_ylabel(f"{name.title()} records")
axes[-1].tick_params(axis="x", rotation=45)
axes[0].set_title(f"{sample.scope.competition_id} · {sample.scope.season} · monthly coverage (UTC)")
fig.tight_layout()
fig.savefig(output_dir / "temporal_coverage.png")
plt.show()
display(pd.DataFrame(report["temporal"]).drop(index="monthly_counts_utc"))

## Observed match targets

Home/draw/away frequencies and goal distributions are descriptive. Missing or unfinished outcomes are excluded explicitly. These frequencies are not a trained baseline and must not be reused as full-sample probabilities for backtesting earlier matches.

In [ ]:
display(pd.DataFrame(report["match_targets"]["outcomes"]).T)
display(pd.DataFrame({
    side: report["match_targets"][f"{side}_goals"] for side in ["home", "away"]
}))
finished = sample.matches.loc[
    sample.matches["status"].eq("finished")
    & sample.matches[["home_goals", "away_goals"]].notna().all(axis=1)
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
outcomes = pd.DataFrame(report["match_targets"]["outcomes"]).T
axes[0].bar(["Home win", "Draw", "Away win"], outcomes["count"], color="#185A80")
axes[0].set_ylabel("Matches")
axes[0].set_title("Observed outcomes")
for side, color in [("home", "#185A80"), ("away", "#B15E24")]:
    freq = finished[f"{side}_goals"].value_counts().sort_index()
    axes[1].plot(freq.index, freq.values, marker="o", label=side.title(), color=color)
axes[1].set_xlabel("Goals scored")
axes[1].set_ylabel("Matches")
axes[1].set_title("Observed goal counts")
axes[1].legend()
fig.tight_layout()
fig.savefig(output_dir / "match_targets.png")
plt.show()

## Valuation skew and repeated observations

All-record histograms weight players by their number of valuation records. The table also reports one latest observation per player, competition, season **and provider**; this is a retrospective sensitivity comparison, not a valid earlier-date feature. Multiple providers remain separate observational units. No outliers are deleted and no target transform is selected from these plots. `log1p(EUR)` is a candidate to compare later using chronological validation; it need not make values normally distributed.

In [ ]:
display(pd.DataFrame(report["valuation_targets"]).T)
target = pd.to_numeric(sample.values["market_value_eur"]).astype(float)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(target.dropna() / 1_000_000, bins=35, color="#185A80", edgecolor="white")
axes[0].set_xlabel("Observed value (million EUR)")
axes[1].hist(np.log1p(target.dropna()), bins=35, color="#B15E24", edgecolor="white")
axes[1].set_xlabel("log(1 + observed value in EUR)")
for ax in axes:
    ax.set_ylabel("Valuation records")
fig.suptitle("All observations · repeated players · source-reported competition context")
fig.tight_layout()
fig.savefig(output_dir / "valuation_targets.png")
plt.show()

## Leakage audit and next steps

- Current-match scores/shots are outcomes, never pre-match inputs. M2 separates its pre-match view and tests current/future outcome independence.
- Full-season totals and latest valuations in EDA are retrospective. A future model must construct features as of each target date.
- Current club, maximum value and other current snapshot fields cannot stand in for historical attributes. Position/nationality in the profile snapshot are not proven date-aligned either.
- M2's history embargo protects event ordering but cannot reconstruct when corrected results were originally published.
- Repeated player valuations are correlated. Use chronological splits and explicitly decide whether the use case is future valuations of known players or unseen-player generalization.
- This sample has already been explored. Establish an untouched later holdout and add earlier seasons before model selection; fit preprocessing on training data only.
- Appearance ingestion, identity reconciliation and historical membership verification remain open data work. See `docs/DATA_QUALITY.md` for measured findings and tracked issues.

No predictive readiness claim follows from a complete CSV or successful EDA.

In [ ]:
report_path = settings.artifact_dir / "m3-quality.json"
report_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
display({"report": str(report_path), "figures": str(output_dir), "models_trained": 0})